#1. Basic Tasks 

##1. Create a bronze table that ingests raw sales data as-is (no transformations), preserving all original columns plus an ingestion timestamp. 

In [0]:
%sql
use catalog cyntexa_dev;
create schema cyntexa_dev.bronze;
create schema cyntexa_dev.silver;
create schema cyntexa_dev.gold;
create or replace table cyntexa_dev.bronze.sales_raw
as
select *,current_timestamp as ingest_ts from read_files('/Volumes/dev/demo/raw/sales.csv')

##2. Build a silver table from bronze that removes duplicates, fixes data types, and drops clearly invalid rows

In [0]:
%sql
create or replace table cyntexa_dev.silver.sales_clean as 
select distinct * from cyntexa_dev.bronze.sales_raw where order_id is not null



##3. Build a gold table that aggregates silver into a business-ready view (e.g., daily revenue by store)

In [0]:
create or replace view cyntexa_dev.gold.daily_revenue as 
select order_date , round(sum(total_amount),2) as total_revenue from cyntexa_dev.silver.sales_clean group by order_date order by order_date


##4. Diagram your bronze/silver/gold pipeline and label, for each layer, who the primary consumer is (engineers, analysts, executives)

###Track data flow:
```
Source Systems (Raw CSV files)
    |
    v
Bronze Layer (Raw Ingestion)
    ├── sales_raw
    ├── Primary Consumer: Data Engineers
    ├── Purpose: Raw data preservation, auditing, reprocessing
    |
    v
Silver Layer (Cleaned & Validated)
    ├── sales_clean
    ├── Primary Consumer: Data Analysts
    ├── Purpose: Clean, deduplicated data for analysis
    |
    v
Gold Layer (Business Aggregates)
    ├── daily_revenue
    ├── Primary Consumer: Business Analysts & Executives
    ├── Purpose: Business-ready KPIs and reports
```


##5. Recreate one part of your silver transformation using Lakeflow Designer's visual, no-code interface and compare the experience to writing it in code. 

**Lakeflow Designer:** excels for rapid, accessible data preparation with built-in previews and a visual audit trail. The code approach offers maximum control and is better suited for complex transformations. In this case, the visual tool actually produced a more thorough null check than the hand-written SQL — demonstrating that no-code tools can improve data quality by making it easier to apply comprehensive rules.

##6. Chain bronze → silver → gold as a Lakeflow Job with proper task dependencies, and configure it to run on a schedule. 

###Job YML

```
resources:
  jobs:
    asssignment_day6_Que_6:
      name: asssignment day6 Que 6
      schedule:
        quartz_cron_expression: 45 54 14 * * ?
        timezone_id: Asia/Calcutta
        pause_status: UNPAUSED
      tasks:
        - task_key: Ingestion_to_bronze
          sql_task:
            file:
              path: "/Workspace/Users/agrawaldeepak386@gmail.com/Cyntexa-assignments/Day 6
                Assignment: Medallion Architecture  & Lakeflow/ingestion_to
                _bronze.sql"
              source: WORKSPACE
            warehouse_id: 01a82f76f6f90d0a
        - task_key: bronze_to_silver
          depends_on:
            - task_key: Ingestion_to_bronze
          sql_task:
            file:
              path: "/Workspace/Users/agrawaldeepak386@gmail.com/Cyntexa-assignments/Day 6
                Assignment: Medallion Architecture  & Lakeflow/bronze to
                silver.sql"
              source: WORKSPACE
            warehouse_id: 01a82f76f6f90d0a
        - task_key: silver_to_gold
          depends_on:
            - task_key: bronze_to_silver
          sql_task:
            file:
              path: "/Workspace/Users/agrawaldeepak386@gmail.com/Cyntexa-assignments/Day 6
                Assignment: Medallion Architecture  & Lakeflow/silver
                to  gold.sql"
              source: WORKSPACE
            warehouse_id: 01a82f76f6f90d0a
      queue:
        enabled: true
      performance_target: PERFORMANCE_OPTIMIZED
```

##7. Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team) and justify why it belongs in gold rather than being computed ad hoc by that team. 

In [0]:

CREATE OR REPLACE VIEW cyntexa_dev.gold.product_inventory_summary AS
SELECT 
    product_id,
    COUNT(DISTINCT order_id) as total_orders,
    SUM(quantity) as total_quantity_sold,
    ROUND(SUM(total_amount), 2) as total_product_revenue,
    ROUND(AVG(total_amount), 2) as avg_unit_price,
    MIN(order_date) as first_sale_date,
    MAX(order_date) as last_sale_date
FROM cyntexa_dev.silver.sales_clean
GROUP BY product_id
ORDER BY total_quantity_sold DESC;
select * from cyntexa_dev.gold.product_inventory_summary

**Justification for Gold Layer Placement:**

This product inventory summary belongs in the gold layer rather than being computed ad hoc for several critical reasons:

1. **Performance & Cost Efficiency**: The aggregation runs once and serves multiple queries, avoiding repeated full table scans. Ad hoc computation would force the inventory team to scan the entire silver table every time they need insights.

2. **Optimized for Business Users**: The inventory team gets a clean, pre-aggregated view with meaningful column names and business-ready formatting, eliminating the need for SQL expertise or repetitive query writing.

3. **Query Simplicity**: The inventory team can run simple `SELECT * FROM gold.product_inventory_summary WHERE product_id = 'X'` queries instead of writing complex aggregations repeatedly.

##8. Write a short design note on which parts of this pipeline should run in the customer's data plane vs. rely on Databricks' control plane, and what that means for a network/security review. 

### Architecture: Control Plane vs Data Plane Separation

| Step | Action | Control Plane (Databricks) | Data Plane (Customer Cloud) |
| --- | --- | --- | --- |
| 1 | Log in to Databricks | Authenticates user and grants access to notebooks, jobs, and clusters. | No action yet. |
| 2 | Start a Cluster | Control Plane sends API request to launch cluster. | EC2 instances (worker nodes) are provisioned in customer's AWS account. |
| 3 | Submit a PySpark Job | Job execution is triggered via Databricks UI. | The Spark job runs on the worker nodes, accessing data from S3. |
| 4 | Read & Process Data | Monitors execution progress and logs status. | Spark reads data from S3, applies transformations, and writes to Delta Lake. |
| 5 | Store Results | Stores metadata about job execution in Databricks workspace. | Transformed data is written to Delta Table in AWS Glue. |
| 6 | Job Completion | Shows job success/failure in Databricks UI. | No further processing unless another job is triggered. |


##9. (Data Analyst) Build a query or lightweight dashboard directly against the gold table, and identify one data-quality issue you can trace back to a specific bronze or silver transformation decision. 

In [0]:
-- Query 1: Daily Revenue Analysis with Data Quality Checks
SELECT 
    order_date,
    total_revenue,
    LAG(total_revenue) OVER (ORDER BY order_date) as prev_day_revenue,
    total_revenue - LAG(total_revenue) OVER (ORDER BY order_date) as revenue_change,
    ROUND((total_revenue - LAG(total_revenue) OVER (ORDER BY order_date)) / LAG(total_revenue) OVER (ORDER BY order_date) * 100, 2) as pct_change
FROM cyntexa_dev.gold.daily_revenue
ORDER BY order_date;

-- Query 2: Check for potential data quality issues - Missing dates
WITH date_range AS (
    SELECT 
        MIN(order_date) as min_date,
        MAX(order_date) as max_date
    FROM cyntexa_dev.gold.daily_revenue
),
expected_dates AS (
    SELECT EXPLODE(SEQUENCE(
        (SELECT min_date FROM date_range),
        (SELECT max_date FROM date_range),
        INTERVAL 1 DAY
    )) as expected_date
)
SELECT 
    ed.expected_date as missing_date
FROM expected_dates ed
LEFT JOIN cyntexa_dev.gold.daily_revenue dr ON ed.expected_date = dr.order_date
WHERE dr.order_date IS NULL
ORDER BY ed.expected_date;

-- Query 3: Investigate root cause - Check silver layer for those missing dates
SELECT 
    order_date,
    COUNT(*) as record_count,
    COUNT(DISTINCT order_id) as unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) as null_order_ids
FROM cyntexa_dev.silver.sales_clean
GROUP BY order_date
ORDER BY order_date;

-- Query 4: Trace back to bronze - Check if data existed before silver transformation
SELECT 
    order_date,
    COUNT(*) as total_records,
    COUNT(DISTINCT order_id) as unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) as null_order_ids,
    COUNT(*) - COUNT(DISTINCT order_id) as duplicate_count
FROM cyntexa_dev.bronze.sales_raw
GROUP BY order_date
ORDER BY order_date;